# Logistic Regression


## 1. Load prepared data and fixed splits

Loads the prepared development bundle through the project's shared splitter. Only train and
validation partitions are used; the locked test partition is never materialized during
development selection.

```python
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
import sys
sys.path.insert(0, str(ROOT))
from src.splitting import load_development_bundle

data, split_meta = load_development_bundle("primary", ROOT)
print("Split:", split_meta["split_id"])
print("Dataset SHA-256:", split_meta["dataset_sha256"])

train = data[data["primary_partition"].eq("train") & data["primary_include"]].copy().reset_index(drop=True)
validation = data[data["primary_partition"].eq("validation") & data["primary_include"]].copy().reset_index(drop=True)

print("Train rows:", len(train))
print("Validation rows:", len(validation))
print("Train label counts:")
print(train["label"].value_counts())
print("Validation label counts:")
print(validation["label"].value_counts())
```


## 2. Build the feature/model pipeline (sparse TF-IDF)

Builds the word unigram/bigram TF-IDF vectorizer fitted on training data only, then transforms
train and validation. The matrix stays sparse — no dense conversion.

```python
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2), min_df=2, max_df=0.99,
    max_features=60000, sublinear_tf=True, strip_accents="unicode",
    dtype=np.float32, lowercase=True,
)

X_train = vectorizer.fit_transform(train["quote_text"].fillna("").astype(str))
X_val = vectorizer.transform(validation["quote_text"].fillna("").astype(str))

print("TF-IDF train shape:", X_train.shape)
print("TF-IDF validation shape:", X_val.shape)
print("Dtype:", X_train.dtype)
```


In [ ]:
import warnings
import time
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)
from sklearn.exceptions import ConvergenceWarning

LABELS = ["Risky", "Neutral", "Safe"]
XTR = X_train
XVA = X_val
YTR = train["label"].to_numpy()
YVA = validation["label"].to_numpy()


In [ ]:
## 3. Train, bounded tune, and predict on validation

Fits the model on training data with bounded hyperparameter tuning over fixed group-respecting
CV folds, then predicts on the held-out validation partition. Convergence warnings are shown.

from sklearn.linear_model import LogisticRegression
MODEL_CLASS = lambda **p: LogisticRegression(random_state=20261102, **p)
GRID = [{{"C": c, "class_weight": cw, "max_iter": 3000, "solver": "lbfgs"}} for c in (0.1, 1.0, 10.0) for cw in (None, "balanced")]

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=20261102)
best = None
for params in GRID:
    model = MODEL_CLASS(**params)
    fold_scores = []
    for fold_idx, (fit_idx, val_idx) in enumerate(skf.split(XTR, YTR)):
        Xf, Xv = XTR[fit_idx], XTR[val_idx]
        yf, yv = YTR[fit_idx], YTR[val_idx]
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.fit(Xf, yf)
            yp = model.predict(Xv)
        for w in caught:
            print(f"Fold {fold_idx} warning: {w.category.__name__}: {w.message}")
        fold_scores.append(f1_score(yv, yp, labels=LABELS, average="macro", zero_division=0))
    mean_score = float(sum(fold_scores) / len(fold_scores))
    print(params, "-> CV macro F1:", round(mean_score, 4))
    if best is None or mean_score > best[0]:
        best = (mean_score, params)

print("Best params:", best[1], "CV macro F1:", round(best[0], 4))

final = MODEL_CLASS(**best[1])
t0 = time.perf_counter()
final.fit(XTR, YTR)
fit_seconds = time.perf_counter() - t0
t0 = time.perf_counter()
y_pred = final.predict(XVA)
predict_seconds = time.perf_counter() - t0

print("Fit seconds:", round(fit_seconds, 3))
print("Predict seconds:", round(predict_seconds, 3))


## 4. Calculate metrics

Computes accuracy, per-class precision/recall/F1/support, macro and weighted averages, the
confusion matrix (counts), and Risky-class false positives / false negatives.

```python
import pandas as pd
import json

report = classification_report(YVA, y_pred, labels=LABELS, output_dict=True, zero_division=0)

summary = {{
    "rows": int(len(YVA)),
    "accuracy": float(accuracy_score(YVA, y_pred)),
    "macro_f1": float(f1_score(YVA, y_pred, labels=LABELS, average="macro", zero_division=0)),
    "macro_precision": float(precision_score(YVA, y_pred, labels=LABELS, average="macro", zero_division=0)),
    "macro_recall": float(recall_score(YVA, y_pred, labels=LABELS, average="macro", zero_division=0)),
    "risky_precision": float(precision_score(YVA, y_pred, labels=["Risky"], average="macro", zero_division=0)),
    "risky_recall": float(recall_score(YVA, y_pred, labels=["Risky"], average="macro", zero_division=0)),
    "risky_false_positives": int(((YVA != "Risky") & (y_pred == "Risky")).sum()),
    "risky_false_negatives": int(((YVA == "Risky") & (y_pred != "Risky")).sum()),
}}
for label in LABELS:
    summary[f"{{label.lower()}}_precision"] = float(report[label]["precision"])
    summary[f"{{label.lower()}}_recall"] = float(report[label]["recall"])
    summary[f"{{label.lower()}}_f1"] = float(report[label]["f1-score"])
    summary[f"{{label.lower()}}_support"] = float(report[label]["support"])

print(json.dumps(summary, indent=2))

cm = confusion_matrix(YVA, y_pred, labels=LABELS)
cm_df = pd.DataFrame(
    cm, index=[f"true_{{l}}" for l in LABELS], columns=[f"pred_{{l}}" for l in LABELS]
)
print(cm_df)
```


In [ ]:
## 5. Plot confusion matrix

Plots the validation confusion matrix as a heatmap. Counts only; row-normalised form is shown
alongside for interpretation.

```python
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
axes[0].set_title("Logistic Regression — Confusion Matrix (counts)")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[1])
axes[1].set_title("Logistic Regression — Confusion Matrix (row-normalised)")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.show()
```


## 6. Interpretation

- Reported scores are on the **validation** partition of the **primary** protocol
  (development data only).
- The locked test partition was not used for tuning or selection.
- Macro F1 treats all three classes equally; Risky precision/recall are reported alongside.
- Coefficients / rules are associations learned from case-derived annotations — not legal
  advice. A missing Risky prediction does not establish that a document is safe.

See `models/logistic_regression/results/primary/metrics.csv` for the saved validation metrics and
`results/comparison/primary/comparison.csv` for the full eight-method comparison.
